In [1]:
import cv2
import numpy as np

def aplicar_modelo_rgbd_hsv(imagen_color, mapa_profundidad):
    """
    Simula la fusión de segmentación por color (HSV) y rango espacial (Depth).
    - El objetivo es detectar un objeto ROJO que esté CERCA (entre 0.5 y 1.5 metros).
    """
    # ----------------- PROCESO 1: Segmentación por Color (HSV) -----------------
    # Convertimos la imagen de BGR (OpenCV) a HSV
    imagen_hsv = cv2.cvtColor(imagen_color, cv2.COLOR_BGR2HSV)
    
    # Definimos los límites para detectar el color rojo (Matiz, Saturación, Valor)
    # Buscaremos rojos muy vivos (alta saturación y brillo)
    rojo_bajo1 = np.array([0, 120, 70])
    rojo_alto1 = np.array([10, 255, 255])
    rojo_bajo2 = np.array([170, 120, 70])
    rojo_alto2 = np.array([180, 255, 255])
    
    # Creamos la máscara para aislar el rojo
    mascara_roja1 = cv2.inRange(imagen_hsv, rojo_bajo1, rojo_alto1)
    mascara_roja2 = cv2.inRange(imagen_hsv, rojo_bajo2, rojo_alto2)
    mascara_color = cv2.bitwise_or(mascara_roja1, mascara_roja2)

    # ----------------- PROCESO 2: Segmentación por Profundidad (D) -----------------
    # Umbral de distancia en mm (por ejemplo: entre 50 cm y 150 cm)
    distancia_minima = 500  
    distancia_maxima = 1500 
    
    # Creamos una máscara de profundidad: solo píxeles dentro del rango de distancia
    mascara_profundidad = cv2.inRange(mapa_profundidad, distancia_minima, distancia_maxima)

    # ----------------- PROCESO 3: Fusión Híbrida (RGB-D + HSV) -----------------
    # AND lógico: El objeto debe ser ROJO "Y" estar en el rango de distancia física
    mascara_final = cv2.bitwise_and(mascara_color, mascara_profundidad)
    
    # Aplicamos la máscara final sobre la imagen original para extraer el objeto
    resultado_objeto = cv2.bitwise_and(imagen_color, imagen_color, mask=mascara_final)
    
    return mascara_color, mascara_profundidad, mascara_final, resultado_objeto

# --- SIMULACIÓN DE DATOS (Para que puedas probar el código sin hardware físico) ---
# Creamos una imagen simulada de 480x640 pixeles (Fondo oscuro)
imagen_prueba = np.zeros((480, 640, 3), dtype=np.uint8)
# Dibujamos un objeto rojo "Cercano" (Caja 1)
cv2.rectangle(imagen_prueba, (100, 150), (250, 300), (0, 0, 255), -1) 
# Dibujamos un objeto rojo "Lejano" (Caja 2)
cv2.rectangle(imagen_prueba, (400, 150), (550, 300), (0, 0, 255), -1) 
# Dibujamos un objeto azul "Cercano" (Distractor)
cv2.rectangle(imagen_prueba, (250, 320), (380, 420), (255, 0, 0), -1) 

# Simulamos el Mapa de Profundidad (en milímetros)
# Por defecto, el "fondo" está muy lejos (5000 mm)
mapa_profundidad_simulado = np.full((480, 640), 5000, dtype=np.uint16)
# La Caja Roja 1 está cerca: 1000 mm (1 metro) -> Debería ser detectada
mapa_profundidad_simulado[150:300, 100:250] = 1000
# La Caja Roja 2 está lejos: 3000 mm (3 metros) -> Debería ser ignorada por distancia
mapa_profundidad_simulado[150:300, 400:550] = 3000
# El objeto azul está cerca: 800 mm -> Debería ser ignorado por color (HSV)
mapa_profundidad_simulado[320:420, 250:380] = 800

# Ejecutamos el modelo
m_color, m_prof, m_final, res = aplicar_modelo_rgbd_hsv(imagen_prueba, mapa_profundidad_simulado)

# Guardamos o mostramos los resultados (puedes visualizarlo en tu entorno de desarrollo)
# cv2.imshow("Original", imagen_prueba)
# cv2.imshow("Filtro Color HSV", m_color)
# cv2.imshow("Filtro Profundidad D", m_prof)
# cv2.imshow("Resultado Hibrido", res)
# cv2.waitKey(0)

In [6]:
import cv2
import numpy as np

# 1. Cargar la fotografía real
imagen = cv2.imread('mi_foto1.jpeg')

# Redimensionar la imagen para que sea más fácil de visualizar en pantalla
imagen = cv2.resize(imagen, (800, 600))
alto, ancho = imagen.shape[:2]

# 2. Segmentación por Color (HSV) - Buscando objetos AZULES
imagen_hsv = cv2.cvtColor(imagen, cv2.COLOR_BGR2HSV)

# Rango ampliado para atrapar azules bajo sombra
# Rango ajustado para AZUL OSCURO (ignorando el celeste)
# Rango ampliado para TODAS las tonalidades de azul (celeste a marino)
azul_bajo = np.array([85, 50, 50])  
azul_alto = np.array([135, 255, 255])
mascara_color = cv2.inRange(imagen_hsv, azul_bajo, azul_alto)

# 3. Simular el Mapa de Profundidad (Canal D)
mapa_profundidad = np.zeros((alto, ancho), dtype=np.uint16)
for y in range(alto):
    # Ajustamos el gradiente para que la zona de la mesa se considere "cerca"
    distancia = int(3000 - (y / alto) * 2000)
    mapa_profundidad[y, :] = distancia

# 4. Segmentación Espacial (Filtro de Profundidad)
# Subimos el umbral máximo a 2600 mm para atrapar las bolsas sobre la mesa
distancia_maxima = 2600
mascara_profundidad = cv2.inRange(mapa_profundidad, 0, distancia_maxima)

# 5. Fusión RGB-D + HSV
# Encontramos la intersección: Píxeles que son AZULES "Y" están CERCA
mascara_final = cv2.bitwise_and(mascara_color, mascara_profundidad)

# 6. Detectar contornos y buscar el objeto más cercano
# Buscamos los bordes de los objetos detectados en la máscara final
contornos, _ = cv2.findContours(mascara_final, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

distancia_mas_corta = float('inf')
contorno_mas_cercano = None

# Analizamos cada objeto encontrado
for contorno in contornos:
    # Ignoramos manchas muy pequeñas (ruido visual)
    if cv2.contourArea(contorno) > 500:
        # Creamos una máscara temporal solo para este objeto específico
        mascara_aislada = np.zeros_like(mascara_final)
        cv2.drawContours(mascara_aislada, [contorno], -1, 255, -1)
        
        # Leemos las distancias de este objeto en el mapa de profundidad
        distancias_del_objeto = mapa_profundidad[mascara_aislada == 255]
        
        if len(distancias_del_objeto) > 0:
            # Encontramos el punto más cercano de este objeto
            distancia_minima_objeto = np.min(distancias_del_objeto)
            
            # Si es el más cercano que hemos visto hasta ahora, lo guardamos como el ganador
            if distancia_minima_objeto < distancia_mas_corta:
                distancia_mas_corta = distancia_minima_objeto
                contorno_mas_cercano = contorno

# 7. Dibujar el recuadro sobre el objeto ganador
imagen_final_marcada = imagen.copy()

if contorno_mas_cercano is not None:
    # Obtener las coordenadas del rectángulo que encierra al contorno
    x, y, w, h = cv2.boundingRect(contorno_mas_cercano)
    
    # Dibujar el rectángulo verde
    cv2.rectangle(imagen_final_marcada, (x, y), (x+w, y+h), (0, 255, 0), 3)
    
    # Añadir un texto indicando la distancia simulada
    texto = f"Mas cercano: {distancia_mas_corta} mm"
    cv2.putText(imagen_final_marcada, texto, (x, y - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

# Mostrar el nuevo resultado
cv2.imshow("5. Objeto mas cercano detectado", imagen_final_marcada)

# --- FUNCIONES DE ESPERA REQUERIDAS POR OPENCV ---
# Esto evita que la ventana diga "No responde"
cv2.waitKey(0)
cv2.destroyAllWindows()

In [5]:
import cv2
import numpy as np

# 1. Cargar la fotografía real
imagen = cv2.imread('mi_foto1.jpeg')

# Redimensionar la imagen para que sea más fácil de visualizar en pantalla
imagen = cv2.resize(imagen, (800, 600))
alto, ancho = imagen.shape[:2]

# 2. Segmentación por Color (HSV) - Buscando objetos AZULES
imagen_hsv = cv2.cvtColor(imagen, cv2.COLOR_BGR2HSV)

# Rango ajustado para AZUL OSCURO (ignorando el celeste)
# Rango ampliado para TODAS las tonalidades de azul (celeste a marino)
azul_bajo = np.array([85, 50, 50])  
azul_alto = np.array([135, 255, 255])
mascara_color = cv2.inRange(imagen_hsv, azul_bajo, azul_alto)

# 3. Simular el Mapa de Profundidad (Canal D)
mapa_profundidad = np.zeros((alto, ancho), dtype=np.uint16)
for y in range(alto):
    # Ajustamos el gradiente para que la zona de la mesa se considere "cerca"
    distancia = int(3000 - (y / alto) * 2000)
    mapa_profundidad[y, :] = distancia

# 4. Segmentación Espacial (Filtro de Profundidad)
# Subimos el umbral máximo a 2600 mm para atrapar las bolsas sobre la mesa
distancia_maxima = 2600
mascara_profundidad = cv2.inRange(mapa_profundidad, 0, distancia_maxima)

# 5. Fusión RGB-D + HSV
# Encontramos la intersección: Píxeles que son AZULES "Y" están CERCA
mascara_final = cv2.bitwise_and(mascara_color, mascara_profundidad)

# 6. Detectar contornos y buscar el objeto más lejano
# Buscamos los bordes de los objetos detectados en la máscara final
contornos, _ = cv2.findContours(mascara_final, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Iniciamos la distancia en 0 para buscar el valor más alto
distancia_mas_lejana = 0 
contorno_mas_lejano = None

# Analizamos cada objeto encontrado
for contorno in contornos:
    # Ignoramos manchas muy pequeñas (ruido visual)
    if cv2.contourArea(contorno) > 500:
        # Creamos una máscara temporal solo para este objeto específico
        mascara_aislada = np.zeros_like(mascara_final)
        cv2.drawContours(mascara_aislada, [contorno], -1, 255, -1)
        
        # Leemos las distancias de este objeto en el mapa de profundidad
        distancias_del_objeto = mapa_profundidad[mascara_aislada == 255]
        
        if len(distancias_del_objeto) > 0:
            # Encontramos el punto MÁS LEJANO (valor máximo) de este objeto
            distancia_maxima_objeto = np.max(distancias_del_objeto)
            
            # Si este número es mayor al que teníamos guardado, este es el nuevo ganador
            if distancia_maxima_objeto > distancia_mas_lejana:
                distancia_mas_lejana = distancia_maxima_objeto
                contorno_mas_lejano = contorno

# 7. Dibujar el recuadro sobre el objeto ganador
imagen_final_marcada = imagen.copy()

if contorno_mas_lejano is not None:
    # Obtener las coordenadas del rectángulo que encierra al contorno
    x, y, w, h = cv2.boundingRect(contorno_mas_lejano)
    
    # Dibujar un rectángulo rojo para indicar "lejano"
    cv2.rectangle(imagen_final_marcada, (x, y), (x+w, y+h), (0, 0, 255), 3)
    
    # Añadir un texto indicando la distancia simulada
    texto = f"Mas lejano: {distancia_mas_lejana} mm"
    cv2.putText(imagen_final_marcada, texto, (x, y - 10), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

# Mostrar el nuevo resultado
cv2.imshow("5. Objeto mas lejano detectado", imagen_final_marcada)

# --- FUNCIONES DE ESPERA AÑADIDAS ---
cv2.waitKey(0)
cv2.destroyAllWindows()